# <span style="color:#00bfff;"> Taller Unidad 2: Búsqueda en Espacio de Estados</span>

**Integrantes del grupo:**

1. [Luis Alberto Alvarado Fonseca]

**Instrucciones del taller:**

- Incluya los bloques de Markdown o de código en Python que considere necesarios. 

- En este taller se promueve el uso responsable del ***"Vibe Coding"***, de manera que asegúrese de argegar **capturas de pantalla** de las respuestas correspondientes al LLM en que se apoye para codificar. Cree una carpeta de "assets" para almacenar las capturas de panatlla y llamarlas dentro del código. Adjunte dicha carpeta en la entrega. 

    *Ejemplo de inserción de una imagen en markdown:* `<div style="width:500px"><img src="assets/miimagen"></div>`

- Es indispensable incluir la **narrativa de las interacciones** con el LLM para validar los procesos lógicos que lo llevaron a su respuesta. (Más allá de la respuesta, correcta o no, se evaluará la *consecución lógica* del cómo llegó a cada conclusión). 

- La entrega del taller debe realizarse en un **archivo comprimido** (.zip, .7z, .rar) que contenga el taller desarrollado (.ipynb) y la carpeta de *assets*. 

## <span style="color:#00bfff;">1. Contexto del Problema</span>
Tras un evento de inundación, un vehículo de rescate autónomo debe evacuar a tres grupos de personas ($G1, G2, G3$) ubicados en puntos críticos y trasladarlos a una *Zona Segura (ZS)*.

**Restricciones del Mundo:**

**1. Capacidad:** El vehículo tiene una capacidad máxima de 2 unidades. El rescatista ocupa 1 unidad permanentemente. Por lo tanto, solo puede transportar a 1 grupo a la vez.

**2. Ubicaciones:** Existen 4 puntos en el mapa: $P1, P2, P3$ y $ZS$.

**3. Estado Inicial:** El rescatista y los tres grupos están en sus puntos de origen ($G1$ en $P1$, $G2$ en $P2$, $G3$ en $P3$). El rescatista inicia en $ZS$.

**4. Seguridad:** Por protocolos de emergencia, el grupo $G1$ y el grupo $G2$ no pueden permanecer solos en la misma ubicación sin la presencia del rescatista (debido a tensiones logísticas previas).

**5. Costos de Desplazamiento:**

- $ZS \leftrightarrow P1: 5$ litros.
- $P1 \leftrightarrow P2: 10$ litros.
- $P2 \leftrightarrow P3: 8$ litros.
- $P3 \leftrightarrow ZS: 15$ litros.


## <span style="color:#00bfff;">2. Definición Formal del Espacio de Estados</span>

Antes de codificar, complete la descripción formal del problema utilizando la notación vista en la Unidad 2.
- **Estado ($S$):** Definir cómo representará la ubicación del rescatista y de cada grupo.
- **Estado Inicial ($s_0$):** Estado de partida.
- **Acciones ($A$):** Movimientos posibles y traslados de personas.
- **Modelo de Transición ($Result(s, a)$):** Reglas de cambio de estado.
- **Test de Objetivo ($Goal$):** Condición de finalización.
- **Costo de Camino ($c$):** Definición de la función de costo.

***<span style="color:#00bfff;">[Ingrese en este bloque Markdown su respuesta]</span>***

<div style="width:500px">
  <img src="https://drive.google.com/thumbnail?id=1-6pSZZ1bN6zT_TSRDMEBsIRrpSPiKzIT&sz=w1000">
</div>

<div style="width:500px">
  <img src="https://drive.google.com/thumbnail?id=1T3wHXPnytrCZ6llrTjgZALJbksLZFEsB&sz=w1000">
</div>

## <span style="color:#00bfff">3. Implementación del Entorno en Python</span>

***Instrucciones:** Complete la siguiente estructura de clase. Puede apoyarse en IA generativa para la sintaxis, pero asegúrese de que las restricciones de seguridad y capacidad estén programadas correctamente.*

In [4]:
class RescateAmbiente:
    def __init__(self):
        # Estado: (Pos_Rescatista, Pos_G1, Pos_G2, Pos_G3)
        self.estado_inicial = ('ZS', 'P1', 'P2', 'P3')
        self.objetivo = ('ZS', 'ZS', 'ZS', 'ZS')
        self.costos = {
            ('ZS', 'P1'): 5,  ('P1', 'ZS'): 5,
            ('P1', 'P2'): 10, ('P2', 'P1'): 10,
            ('P2', 'P3'): 8,  ('P3', 'P2'): 8,
            ('P3', 'ZS'): 15, ('ZS', 'P3'): 15
        }
        # Solo tramos directos del mapa
        self.adyacentes = {
            'ZS': ['P1', 'P3'],
            'P1': ['ZS', 'P2'],
            'P2': ['P1', 'P3'],
            'P3': ['P2', 'ZS']
        }

    def es_seguro(self, estado):
        """
        G1 y G2 NO pueden estar en la misma ubicacion
        si el rescatista no esta presente ahi.
        """
        rescatista, g1, g2, g3 = estado
        if g1 == g2 and g1 != rescatista:
            return False
        return True

    def acciones_legales(self, estado):
        """
        Retorna lista de acciones validas desde un estado:
          ('Mover_Solo', destino)
          ('Transportar', 'GX', destino)
        """
        rescatista, g1, g2, g3 = estado
        acciones = []
        grupos = [('G1', g1), ('G2', g2), ('G3', g3)]

        for destino in self.adyacentes[rescatista]:
            # Mover solo
            nuevo_estado = (destino, g1, g2, g3)
            if self.es_seguro(nuevo_estado):
                acciones.append(('Mover_Solo', destino))

            # Transportar un grupo (capacidad: solo 1 a la vez)
            for nombre, pos in grupos:
                if pos == rescatista:
                    nuevo_g = {'G1': g1, 'G2': g2, 'G3': g3}
                    nuevo_g[nombre] = destino
                    nuevo_estado = (destino, nuevo_g['G1'], nuevo_g['G2'], nuevo_g['G3'])
                    if self.es_seguro(nuevo_estado):
                        acciones.append(('Transportar', nombre, destino))

        return acciones

    def aplicar_accion(self, estado, accion):
        """Retorna el nuevo estado resultante de aplicar una accion."""
        rescatista, g1, g2, g3 = estado
        if accion[0] == 'Mover_Solo':
            _, destino = accion
            return (destino, g1, g2, g3)
        elif accion[0] == 'Transportar':
            _, grupo, destino = accion
            nuevo_g = {'G1': g1, 'G2': g2, 'G3': g3}
            nuevo_g[grupo] = destino
            return (destino, nuevo_g['G1'], nuevo_g['G2'], nuevo_g['G3'])

    def es_objetivo(self, estado):
        """Los tres grupos deben estar en ZS."""
        _, g1, g2, g3 = estado
        return g1 == 'ZS' and g2 == 'ZS' and g3 == 'ZS'

    def costo_accion(self, estado, accion):
        """Retorna el costo en litros del desplazamiento."""
        rescatista = estado[0]
        destino = accion[1] if accion[0] == 'Mover_Solo' else accion[2]
        return self.costos[(rescatista, destino)]


# ============================================================
#                  VERIFICACIÓN DEL ENTORNO
# ============================================================
env = RescateAmbiente()

print("=" * 60)
print("   VERIFICACIÓN DEL ENTORNO RescateAmbiente")
print("=" * 60)

# --- 1. ESTADO INICIAL ---
print("\n📍 ESTADO INICIAL")
print(f"   Estado : {env.estado_inicial}")
print(f"   Formato: (Rescatista, G1, G2, G3)")
print(f"   ¿Es objetivo? → {env.es_objetivo(env.estado_inicial)}")
print(f"   ¿Es seguro?   → {env.es_seguro(env.estado_inicial)}")

# --- 2. ACCIONES LEGALES DESDE ESTADO INICIAL ---
print("\n🔵 ACCIONES LEGALES DESDE ESTADO INICIAL")
acciones = env.acciones_legales(env.estado_inicial)
for a in acciones:
    print(f"   {a}  →  costo: {env.costo_accion(env.estado_inicial, a)} L")

# --- 3. TEST RESTRICCIÓN DE SEGURIDAD ---
print("\n" + "=" * 60)
print("🔒 TEST RESTRICCIÓN DE SEGURIDAD (G1 y G2 no pueden estar")
print("   solos en la misma ubicacion sin el rescatista)")
print("=" * 60)

casos = [
    ('P1', 'P1', 'P1', 'P3'),  # Rescatista con G1 y G2 → SEGURO
    ('ZS', 'P1', 'P1', 'P3'),  # G1 y G2 solos en P1   → INSEGURO
    ('P2', 'P1', 'P1', 'P3'),  # G1 y G2 solos en P1   → INSEGURO
    ('ZS', 'ZS', 'P2', 'P3'),  # G1 y G2 en distintos  → SEGURO
    ('ZS', 'ZS', 'ZS', 'P3'),  # Rescatista con G1, G2 en ZS → SEGURO
    ('P1', 'ZS', 'ZS', 'P3'),  # G1 y G2 solos en ZS   → INSEGURO
]
for caso in casos:
    r, g1, g2, g3 = caso
    seguro = env.es_seguro(caso)
    icono = "✅ SEGURO  " if seguro else "❌ INSEGURO"
    print(f"   {icono} | R={r}  G1={g1}  G2={g2}  G3={g3}")

# --- 4. TEST CAPACIDAD ---
print("\n" + "=" * 60)
print("🚗 TEST CAPACIDAD (máximo 1 grupo por viaje)")
print("=" * 60)
estado_cap = ('P1', 'P1', 'P2', 'P1')  # G1 y G3 en P1 con rescatista
print(f"   Estado de prueba: R=P1 | G1=P1 | G2=P2 | G3=P1")
print(f"   (G1 y G3 disponibles para transportar, solo puede ir 1)")
for a in env.acciones_legales(estado_cap):
    nuevo = env.aplicar_accion(estado_cap, a)
    print(f"   {a}  →  {nuevo}")

# --- 5. TEST ESTADO OBJETIVO ---
print("\n" + "=" * 60)
print("🏁 TEST ESTADO OBJETIVO")
print("=" * 60)
casos_obj = [
    ('ZS', 'ZS', 'ZS', 'ZS'),  # Todos en ZS → objetivo
    ('ZS', 'ZS', 'ZS', 'P3'),  # G3 pendiente → no objetivo
    ('P1', 'ZS', 'ZS', 'ZS'),  # Rescatista fuera, grupos en ZS → objetivo igual
]
for caso in casos_obj:
    print(f"   Estado {caso} → ¿Es objetivo? {env.es_objetivo(caso)}")

# --- 6. SIMULACIÓN PASO A PASO ---
print("\n" + "=" * 60)
print("🔄 SIMULACIÓN: Rescate de G1 (primeros 3 pasos)")
print("=" * 60)
estado = env.estado_inicial
costo_total = 0
pasos = [
    ('Mover_Solo', 'P1'),
    ('Transportar', 'G1', 'ZS'),
    ('Mover_Solo', 'P1'),
]
for i, paso in enumerate(pasos, 1):
    legales = env.acciones_legales(estado)
    if paso in legales:
        costo = env.costo_accion(estado, paso)
        costo_total += costo
        estado = env.aplicar_accion(estado, paso)
        print(f"   Paso {i}: {paso}")
        print(f"           Costo: {costo}L | Estado: {estado}")
    else:
        print(f"   Paso {i}: ❌ Accion {paso} NO legal desde {estado}")

print(f"\n   Costo acumulado : {costo_total} litros")
print(f"   ¿Es objetivo?   : {env.es_objetivo(estado)}")
print("\n" + "=" * 60)
print("   FIN DE VERIFICACIÓN")
print("=" * 60)

   VERIFICACIÓN DEL ENTORNO RescateAmbiente

📍 ESTADO INICIAL
   Estado : ('ZS', 'P1', 'P2', 'P3')
   Formato: (Rescatista, G1, G2, G3)
   ¿Es objetivo? → False
   ¿Es seguro?   → True

🔵 ACCIONES LEGALES DESDE ESTADO INICIAL
   ('Mover_Solo', 'P1')  →  costo: 5 L
   ('Mover_Solo', 'P3')  →  costo: 15 L

🔒 TEST RESTRICCIÓN DE SEGURIDAD (G1 y G2 no pueden estar
   solos en la misma ubicacion sin el rescatista)
   ✅ SEGURO   | R=P1  G1=P1  G2=P1  G3=P3
   ❌ INSEGURO | R=ZS  G1=P1  G2=P1  G3=P3
   ❌ INSEGURO | R=P2  G1=P1  G2=P1  G3=P3
   ✅ SEGURO   | R=ZS  G1=ZS  G2=P2  G3=P3
   ✅ SEGURO   | R=ZS  G1=ZS  G2=ZS  G3=P3
   ❌ INSEGURO | R=P1  G1=ZS  G2=ZS  G3=P3

🚗 TEST CAPACIDAD (máximo 1 grupo por viaje)
   Estado de prueba: R=P1 | G1=P1 | G2=P2 | G3=P1
   (G1 y G3 disponibles para transportar, solo puede ir 1)
   ('Mover_Solo', 'ZS')  →  ('ZS', 'P1', 'P2', 'P1')
   ('Transportar', 'G1', 'ZS')  →  ('ZS', 'ZS', 'P2', 'P1')
   ('Transportar', 'G3', 'ZS')  →  ('ZS', 'P1', 'P2', 'ZS')
   ('Mov

***<span style="color:#00bfff;">[Incluya aquí la narrativa de sus interacciones con IA de los aspectos más relevantes para llegar a la solución.] </span>***
<div style="width:500px">
  <img src="https://drive.google.com/thumbnail?id=14f55miKQR2QEp6ehtQF8pm84X4cAK7pz&sz=w1000">
</div>

<div style="width:500px">
  <img src="https://drive.google.com/thumbnail?id=1JeDgYOL7xaKoJB5rgxZkCd24FS8h5_dP&sz=w1000">
</div>


## <span style="color:#00bfff;">4. Algoritmos de Búsqueda y Análisis </span>

**Parte A: Búsqueda No Informada (DFS)**

Implemente el algoritmo **Depth-First Search**.
- Pregunta: ¿Cómo afecta el orden de las acciones en la lista de acciones_legales al resultado de DFS en este problema?
- Análisis: Muestre la ruta encontrada y el costo total de combustible.

**Parte B: Búsqueda Informada (A\*)** 

Diseñe una función heurística que estime el costo de combustible restante para que todos los grupos lleguen a la Zona Segura (ZS).
- **Requisito de IA:** Solicite a una IA generativa (Gemini, ChatGPT, etc.) que proponga una heurística para este problema de rescate.
- **Evaluación Crítica:** Pegue el prompt utilizado y la respuesta de la IA. Luego, demuestre matemáticamente si la heurística propuesta es admisible ($h(n) \leq h^*(n)$).

In [9]:
# Incluya aquí su implementación de Depth-First Search 


***<span style="color:#00bfff;">[Ingrese en este bloque Markdown su respuesta]</span>***



In [8]:
class RescateAmbiente:
    def __init__(self):
        self.estado_inicial = ('ZS', 'P1', 'P2', 'P3')
        self.objetivo = ('ZS', 'ZS', 'ZS', 'ZS')
        self.costos = {
            ('ZS', 'P1'): 5,  ('P1', 'ZS'): 5,
            ('P1', 'P2'): 10, ('P2', 'P1'): 10,
            ('P2', 'P3'): 8,  ('P3', 'P2'): 8,
            ('P3', 'ZS'): 15, ('ZS', 'P3'): 15
        }
        self.adyacentes = {
            'ZS': ['P1', 'P3'],
            'P1': ['ZS', 'P2'],
            'P2': ['P1', 'P3'],
            'P3': ['P2', 'ZS']
        }

    def es_seguro(self, estado):
        rescatista, g1, g2, g3 = estado
        if g1 == g2 and g1 != rescatista:
            return False
        return True

    def acciones_legales(self, estado):
        rescatista, g1, g2, g3 = estado
        acciones = []
        grupos = [('G1', g1), ('G2', g2), ('G3', g3)]
        for destino in self.adyacentes[rescatista]:
            nuevo_estado = (destino, g1, g2, g3)
            if self.es_seguro(nuevo_estado):
                acciones.append(('Mover_Solo', destino))
            for nombre, pos in grupos:
                if pos == rescatista:
                    nuevo_g = {'G1': g1, 'G2': g2, 'G3': g3}
                    nuevo_g[nombre] = destino
                    nuevo_estado = (destino, nuevo_g['G1'], nuevo_g['G2'], nuevo_g['G3'])
                    if self.es_seguro(nuevo_estado):
                        acciones.append(('Transportar', nombre, destino))
        return acciones

    def aplicar_accion(self, estado, accion):
        rescatista, g1, g2, g3 = estado
        if accion[0] == 'Mover_Solo':
            return (accion[1], g1, g2, g3)
        elif accion[0] == 'Transportar':
            nuevo_g = {'G1': g1, 'G2': g2, 'G3': g3}
            nuevo_g[accion[1]] = accion[2]
            return (accion[2], nuevo_g['G1'], nuevo_g['G2'], nuevo_g['G3'])

    def es_objetivo(self, estado):
        _, g1, g2, g3 = estado
        return g1 == 'ZS' and g2 == 'ZS' and g3 == 'ZS'

    def costo_accion(self, estado, accion):
        rescatista = estado[0]
        destino = accion[1] if accion[0] == 'Mover_Solo' else accion[2]
        return self.costos[(rescatista, destino)]


# ============================================================
# PARTE A: DFS — Depth First Search (Búsqueda No Informada)
# Estructura LIFO: expande siempre el nodo más reciente.
# No garantiza optimalidad en costo.
# ============================================================
def dfs(env):
    pila = [(env.estado_inicial, [], 0)]
    visitados = set()
    nodos_explorados = 0

    while pila:
        estado, camino, costo = pila.pop()  # LIFO

        if estado in visitados:
            continue
        visitados.add(estado)
        nodos_explorados += 1

        if env.es_objetivo(estado):
            return camino, costo, nodos_explorados

        # reversed() para explorar primero el primer elemento de acciones_legales
        for accion in reversed(env.acciones_legales(estado)):
            nuevo_estado = env.aplicar_accion(estado, accion)
            if nuevo_estado not in visitados:
                nuevo_costo = costo + env.costo_accion(estado, accion)
                pila.append((nuevo_estado, camino + [accion], nuevo_costo))

    return None, float('inf'), nodos_explorados


env = RescateAmbiente()

print("=" * 65)
print("   PARTE A: BÚSQUEDA NO INFORMADA — DFS")
print("=" * 65)
camino_dfs, costo_dfs, nodos_dfs = dfs(env)

print(f"\n📌 Nodos explorados : {nodos_dfs}")
print(f"💧 Costo total      : {costo_dfs} litros")
print(f"\n🗺️  Ruta encontrada por DFS ({len(camino_dfs)} pasos):")
estado = env.estado_inicial
print(f"   Estado inicial: {estado}")
for i, accion in enumerate(camino_dfs, 1):
    costo_paso = env.costo_accion(estado, accion)
    estado = env.aplicar_accion(estado, accion)
    print(f"   Paso {i:02d}: {str(accion):<35} | +{costo_paso:2d}L | → {estado}")
print(f"\n   ¿Estado final es objetivo? {env.es_objetivo(estado)}")

print("\n" + "=" * 65)
print("   ANÁLISIS: Efecto del orden de acciones_legales en DFS")
print("=" * 65)
print("""
   DFS expande el ÚLTIMO nodo en la pila (LIFO).
   El orden de acciones_legales define qué rama se toma primero:

   → 'Mover_Solo' antes que 'Transportar':
     DFS prefiere moverse sin grupos → rutas más largas y costosas.

   → 'Transportar' primero:
     DFS intenta llevar grupos de inmediato → rutas más cortas.

   Conclusión: DFS encontró solución válida pero NO óptima.
   El costo depende directamente del orden de exploración.
""")

   PARTE A: BÚSQUEDA NO INFORMADA — DFS

📌 Nodos explorados : 63
💧 Costo total      : 418 litros

🗺️  Ruta encontrada por DFS (56 pasos):
   Estado inicial: ('ZS', 'P1', 'P2', 'P3')
   Paso 01: ('Mover_Solo', 'P1')                | + 5L | → ('P1', 'P1', 'P2', 'P3')
   Paso 02: ('Transportar', 'G1', 'ZS')         | + 5L | → ('ZS', 'ZS', 'P2', 'P3')
   Paso 03: ('Mover_Solo', 'P1')                | + 5L | → ('P1', 'ZS', 'P2', 'P3')
   Paso 04: ('Mover_Solo', 'P2')                | +10L | → ('P2', 'ZS', 'P2', 'P3')
   Paso 05: ('Transportar', 'G2', 'P1')         | +10L | → ('P1', 'ZS', 'P1', 'P3')
   Paso 06: ('Mover_Solo', 'ZS')                | + 5L | → ('ZS', 'ZS', 'P1', 'P3')
   Paso 07: ('Transportar', 'G1', 'P1')         | + 5L | → ('P1', 'P1', 'P1', 'P3')
   Paso 08: ('Transportar', 'G2', 'ZS')         | + 5L | → ('ZS', 'P1', 'ZS', 'P3')
   Paso 09: ('Mover_Solo', 'P1')                | + 5L | → ('P1', 'P1', 'ZS', 'P3')
   Paso 10: ('Transportar', 'G1', 'ZS')         | + 5L | → ('Z

***<span style="color:#00bfff;">[Incluya aquí la narrativa de sus interacciones con IA de los aspectos más relevantes para llegar a la solución.] </span>***
<div style="width:500px">
  <img src="https://drive.google.com/thumbnail?id=1pIk-RlyDFThuwxfpgXMoCAZEO6VSsGBj&sz=w1000">
</div>

In [ ]:
# Incluya aquí su implmentación de A*
***<span style="color:#00bfff;">[Ingrese en este bloque Markdown su respuesta]</span>***



## <span style="color: #00bfff"> 5. Evaluación de desempeño</span>

Presente una tabla comparativa entre los dos algoritmos:

| Métrica | DFS | A* |
|--:|:--:|:--:|
| Nodos expandidos |<span style="color: #00bfff">*Escriba su respuesta aquí*</span> |<span style="color: #00bfff">*Escriba su respuesta aquí*</span> |
| Costo de la solución (combustible) | <span style="color: #00bfff">*Escriba su respuesta aquí*</span>| <span style="color: #00bfff">*Escriba su respuesta aquí*</span>| 
| Tiempo de ejecución | <span style="color: #00bfff">*Escriba su respuesta aquí*</span>| <span style="color: #00bfff">*Escriba su respuesta aquí*</span>|

## <span style="color:#00bfff;"> 6. Rúbrica de Evaluación</span>

| Criterio | Excelente (5.0) | Bueno (3.5 - 4.5) | Insuficiente (< 3.5) | Peso |
| :--- | :--- | :--- | :--- | :---: |
| **Narrativa y Evidencias (Vibe Coding)** | Documenta de forma exhaustiva el diálogo con el LLM. Las capturas de pantalla en la carpeta *assets* son claras y están vinculadas. Explica la intención de cada prompt y cómo validó o corrigió la respuesta de la IA. | Incluye narrativa y capturas, pero la secuencia lógica del diálogo es intermitente. La explicación de los ajustes manuales al código generado es superficial. | No hay una narrativa coherente. Faltan capturas de pantalla o el acceso a la carpeta de *assets*. No se explican los procesos lógicos tras los prompts. | **40%** |
| **Implementación y Depuración** | El código es funcional y resuelve el problema respetando todas las restricciones. Demuestra entendimiento al corregir errores lógicos (bugs) y optimizar las funciones de seguridad. | El código funciona pero depende de la salida de la IA sin personalización. Presenta dificultades menores en la gestión de restricciones o consistencia de costos. | El código no es funcional o ignora restricciones críticas. No hay evidencia de depuración manual del código entregado por la IA. | **20%** |
| **Modelado Formal del Espacio** | Define con precisión el quíntuplo formal $\mathcal{P} = \langle S, s_0, A, R, G, C \rangle$ en LaTeX. Existe correlación total entre la definición matemática y la estructura de datos. | El modelado formal es correcto, pero existen discrepancias menores entre la definición teórica y la lógica aplicada en el desarrollo del entorno. | El modelado es genérico, incompleto o no utiliza la notación formal requerida para los estándares de la asignatura. | **15%** |
| **Algoritmos de Búsqueda** | Implementa DFS y A* correctamente. Presenta una tabla comparativa rigurosa (nodos, costos, tiempo) y analiza los resultados con base en la teoría de la Unidad 2. | Implementa los algoritmos, pero el análisis es descriptivo y no utiliza las métricas para justificar técnicamente el comportamiento de la búsqueda. | No logra implementar ambos algoritmos o los resultados obtenidos son incoherentes con el espacio de búsqueda y costos definidos. | **15%** |
| **Análisis de Heurística** | Evalúa críticamente la heurística sugerida por la IA y demuestra, mediante rigor lógico o contraejemplos, su admisibilidad ($h(n) \leq h^*(n)$). | La heurística es funcional, pero la justificación de admisibilidad es vaga o se acepta la sugerencia de la IA sin un proceso de verificación propio. | La heurística no es admisible (sobreestima el costo) o no se presenta un análisis sobre su impacto en la eficiencia de la búsqueda. | **10%** |